# 02 — Data Quality & Cleaning (PySpark)

## Objective
Identify and fix all data quality issues across all five
tables using PySpark distributed operations — no pandas,
no row-by-row loops, no .collect() on large DataFrames.

## What We Fix
- Duplicate rows
- Impossible values (age, income, amounts)
- Date string → proper DateType casting
- Null imputation strategies per column
- Outlier capping using percentile bounds
- Standardizing categorical values (city names, segments)

## New PySpark Concepts in This Notebook
- F.when().otherwise() for conditional imputation
- F.percentile_approx() for outlier bounds inside groupBy
- Window functions for forward-fill within customer series
- .fillna() with different values per column
- Writing cleaned data to Parquet (faster than CSV for Spark)

## The Golden Rule
Never call .collect() or .toPandas() on a large DataFrame.
Only call these after aggregation when result is small.

In [ ]:
def read_table(name):
    return (
        spark.read.format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

customers   = read_table("customers")
products    = read_table("products")
txn_history = read_table("transactions_history")
clv_labels  = read_table("clv_labels")

print("All tables loaded ✅")
print(f"  customers   : {customers.count():,}")
print(f"  products    : {products.count():,}")
print(f"  txn_history : {txn_history.count():,}")
print(f"  clv_labels  : {clv_labels.count():,}")

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    DateType, BooleanType, DoubleType,
    IntegerType, StringType
)
import os

# Must be set before SparkSession is created
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_PATH   = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")
DB_PATH    = r"D:\some\other\drive\finance_clv.db"
DB_URL     = f"jdbc:sqlite:{DB_PATH}"

LOCAL_TEMP = os.path.abspath("../data/spark_temp")
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("CLV_Data_Quality")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version  : {spark.version}")
print(f"HADOOP_HOME    : {os.environ.get('HADOOP_HOME')}")
print(f"Temp dir       : {LOCAL_TEMP}")
print("SparkSession ready ✅")

c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version : 4.2.0
Temp dir      : c:\Users\dsp96\Desktop\realworld-ds-ml\finance\personal_finance_clv\data\spark_temp
SparkSession ready ✅


In [2]:
print("STEP 1 — REMOVE DUPLICATES\n")

before = {
    "customers"  : customers.count(),
    "products"   : products.count(),
    "txn_history": txn_history.count(),
    "clv_labels" : clv_labels.count(),
}

# Remove exact duplicate rows first
customers   = customers.dropDuplicates()
products    = products.dropDuplicates()
txn_history = txn_history.dropDuplicates()
clv_labels  = clv_labels.dropDuplicates()

# Remove primary key duplicates — keep first occurrence
customers   = customers.dropDuplicates(["customer_id"])
products    = products.dropDuplicates(["product_id"])
txn_history = txn_history.dropDuplicates(["txn_id"])
clv_labels  = clv_labels.dropDuplicates(["customer_id"])

after = {
    "customers"  : customers.count(),
    "products"   : products.count(),
    "txn_history": txn_history.count(),
    "clv_labels" : clv_labels.count(),
}

print(f"{'Table':<15} {'Before':>10} {'After':>10} {'Removed':>10}")
print("-" * 48)
for name in before:
    removed = before[name] - after[name]
    print(f"  {name:<13} {before[name]:>10,} "
          f"{after[name]:>10,} {removed:>10,}")

STEP 1 — REMOVE DUPLICATES



NameError: name 'customers' is not defined

In [ ]:
print("STEP 2 — CAST DATE COLUMNS\n")

# Fix customers
customers = customers.withColumn(
    "account_open_date",
    F.to_date(F.col("account_open_date"), "yyyy-MM-dd")
)

# Fix products
products = products.withColumn(
    "open_date",
    F.to_date(F.col("open_date"), "yyyy-MM-dd")
)

# Fix transactions — also extract year, month, day
txn_history = txn_history.withColumn(
    "txn_date",
    F.to_date(F.col("txn_date"), "yyyy-MM-dd")
).withColumn(
    "txn_year",  F.year("txn_date")
).withColumn(
    "txn_month", F.month("txn_date")
).withColumn(
    "txn_day",   F.dayofmonth("txn_date")
).withColumn(
    "txn_dayofweek", F.dayofweek("txn_date")
).withColumn(
    "is_weekend",
    (F.dayofweek("txn_date").isin([1, 7])).cast(IntegerType())
)

# Verify — count null dates after conversion
# If format didn't match, to_date returns null silently
null_dates = txn_history.filter(
    F.col("txn_date").isNull()
).count()

print(f"  Null txn_date after conversion : {null_dates:,}")
print(f"  (should be 0 — format must match)")

print("\nSample with new date columns:")
txn_history.select(
    "txn_id", "txn_date", "txn_year",
    "txn_month", "txn_dayofweek", "is_weekend"
).show(5)

STEP 2 — CAST DATE COLUMNS

  Null txn_date after conversion : 0
  (should be 0 — format must match)

Sample with new date columns:
+-------------+----------+--------+---------+-------------+----------+
|       txn_id|  txn_date|txn_year|txn_month|txn_dayofweek|is_weekend|
+-------------+----------+--------+---------+-------------+----------+
|TXN0000000001|2023-01-02|    2023|        1|            2|         0|
|TXN0000000006|2023-01-28|    2023|        1|            7|         1|
|TXN0000000014|2023-01-04|    2023|        1|            4|         0|
|TXN0000000020|2023-01-01|    2023|        1|            1|         1|
|TXN0000000023|2023-01-28|    2023|        1|            7|         1|
+-------------+----------+--------+---------+-------------+----------+
only showing top 5 rows


In [ ]:
print("STEP 3 — FIX BOOLEAN AND INTEGER COLUMNS\n")

# Customers — convert 0/1 integers to proper boolean
bool_cols_customers = [
    "kyc_complete", "pan_linked", "aadhaar_linked",
    "rm_assigned", "is_nri"
]
for col in bool_cols_customers:
    customers = customers.withColumn(
        col, F.col(col).cast(BooleanType())
    )

# Products
products = products.withColumn(
    "is_active", F.col("is_active").cast(BooleanType())
)

# Transactions
txn_history = txn_history.withColumn(
    "is_international",
    F.col("is_international").cast(BooleanType())
)

# Verify schema after casting
print("Customers schema (key columns):")
customers.select(
    "customer_id", "kyc_complete", "pan_linked",
    "rm_assigned", "monthly_income", "cibil_score"
).printSchema()

print("Boolean value distribution (customers):")
customers.select(
    F.sum(F.col("kyc_complete").cast(IntegerType())).alias("kyc_complete_count"),
    F.sum(F.col("pan_linked").cast(IntegerType())).alias("pan_linked_count"),
    F.sum(F.col("rm_assigned").cast(IntegerType())).alias("rm_assigned_count"),
    F.count("customer_id").alias("total_customers")
).show()

STEP 3 — FIX BOOLEAN AND INTEGER COLUMNS

Customers schema (key columns):
root
 |-- customer_id: string (nullable = true)
 |-- kyc_complete: boolean (nullable = true)
 |-- pan_linked: boolean (nullable = true)
 |-- rm_assigned: boolean (nullable = true)
 |-- monthly_income: double (nullable = true)
 |-- cibil_score: double (nullable = true)

Boolean value distribution (customers):


+------------------+----------------+-----------------+---------------+
|kyc_complete_count|pan_linked_count|rm_assigned_count|total_customers|
+------------------+----------------+-----------------+---------------+
|             44556|           47455|             7407|          50000|
+------------------+----------------+-----------------+---------------+



In [ ]:
print("STEP 4 — FIX IMPOSSIBLE VALUES\n")

# ── Customers ─────────────────────────────────────────────────
print("── Customers ──")

# Age: must be 18-100
impossible_age = customers.filter(
    (F.col("age") < 18) | (F.col("age") > 100)
).count()
print(f"  Impossible age values   : {impossible_age:,}")

customers = customers.withColumn(
    "age",
    F.when(
        (F.col("age") < 18) | (F.col("age") > 100), None
    ).otherwise(F.col("age"))
)

# CIBIL score: must be 300-900 or null (no credit history)
impossible_cibil = customers.filter(
    F.col("cibil_score").isNotNull() &
    ((F.col("cibil_score") < 300) | (F.col("cibil_score") > 900))
).count()
print(f"  Impossible CIBIL values : {impossible_cibil:,}")

customers = customers.withColumn(
    "cibil_score",
    F.when(
        F.col("cibil_score").isNotNull() &
        ((F.col("cibil_score") < 300) | (F.col("cibil_score") > 900)),
        None
    ).otherwise(F.col("cibil_score"))
)

# Monthly income: must be positive
customers = customers.withColumn(
    "monthly_income",
    F.when(F.col("monthly_income") <= 0, None)
     .otherwise(F.col("monthly_income"))
)

# ── Transactions ───────────────────────────────────────────────
print("\n── Transactions ──")

# Amount: must be positive
neg_amounts = txn_history.filter(F.col("amount") <= 0).count()
print(f"  Negative/zero amounts   : {neg_amounts:,}")

txn_history = txn_history.filter(F.col("amount") > 0)

# Balance after cannot be negative (floor at 0)
txn_history = txn_history.withColumn(
    "balance_after",
    F.when(F.col("balance_after") < 0, 0.0)
     .otherwise(F.col("balance_after"))
)

print("\nAfter fixing impossible values:")
print(f"  txn_history rows: {txn_history.count():,}")

STEP 4 — FIX IMPOSSIBLE VALUES

── Customers ──
  Impossible age values   : 1,074
  Impossible CIBIL values : 2,701

── Transactions ──
  Negative/zero amounts   : 0

After fixing impossible values:
  txn_history rows: 16,608,362


In [ ]:
print("STEP 5 — NULL IMPUTATION\n")

# ── Customers ─────────────────────────────────────────────────
print("── Customers ──")

# Age → median by segment
age_medians = customers.groupBy("segment").agg(
    F.percentile_approx("age", 0.5, 1000).alias("median_age")
)
print("  Median age by segment:")
age_medians.show()

customers = customers.join(
    age_medians, on="segment", how="left"
).withColumn(
    "age",
    F.coalesce(F.col("age"), F.col("median_age"))
).drop("median_age")

# CIBIL score → median by segment
cibil_medians = customers.groupBy("segment").agg(
    F.percentile_approx("cibil_score", 0.5, 1000).alias("median_cibil")
)
customers = customers.join(
    cibil_medians, on="segment", how="left"
).withColumn(
    "cibil_score",
    F.coalesce(F.col("cibil_score"), F.col("median_cibil"))
).drop("median_cibil")

# Gender → "Not Specified"
customers = customers.fillna(
    {"gender": "Not Specified"}
)

# Monthly income → median by segment (for the rare null)
income_medians = customers.groupBy("segment").agg(
    F.percentile_approx("monthly_income", 0.5, 1000).alias("median_income")
)
customers = customers.join(
    income_medians, on="segment", how="left"
).withColumn(
    "monthly_income",
    F.coalesce(F.col("monthly_income"), F.col("median_income"))
).drop("median_income")

# ── Transactions ───────────────────────────────────────────────
print("── Transactions ──")

# merchant_category and channel → "unknown"
txn_history = txn_history.fillna({
    "merchant_category": "unknown",
    "channel"          : "unknown",
})

# Verify
remaining_nulls = txn_history.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["merchant_category", "channel", "amount"]
]).collect()[0]

print(f"\n  merchant_category nulls : {remaining_nulls['merchant_category']}")
print(f"  channel nulls           : {remaining_nulls['channel']}")
print(f"  amount nulls            : {remaining_nulls['amount']}")
print(f"  (all should be 0)")

STEP 5 — NULL IMPUTATION

── Customers ──


  Median age by segment:
+---------------+----------+
|        segment|median_age|
+---------------+----------+
| salaried_entry|        29|
|  self_employed|        42|
|        retired|        69|
|   salaried_mid|        39|
|        student|        21|
|salaried_senior|        48|
|            hni|        47|
+---------------+----------+

── Transactions ──

  merchant_category nulls : 0
  channel nulls           : 0
  amount nulls            : 0
  (all should be 0)


In [ ]:
print("STEP 6 — OUTLIER CAPPING\n")

# ── Transaction amounts ───────────────────────────────────────
print("── Transaction Amount Capping ──")

quantiles = txn_history.approxQuantile(
    "amount", [0.001, 0.999], relativeError=0.01
)
lower_bound = quantiles[0]
upper_bound = quantiles[1]

print(f"  Lower bound (P0.1)  : ₹{lower_bound:>12,.2f}")
print(f"  Upper bound (P99.9) : ₹{upper_bound:>12,.2f}")

# Cap — no .count() after this, it is a transformation only
txn_history = txn_history.withColumn(
    "amount",
    F.when(F.col("amount") < lower_bound,  lower_bound)
     .when(F.col("amount") > upper_bound,  upper_bound)
     .otherwise(F.col("amount"))
)

print("  Amount capping applied ✅ (lazy — no shuffle triggered)")

# ── Customer income ───────────────────────────────────────────
print("\n── Monthly Income Capping ──")

income_quantiles = customers.approxQuantile(
    "monthly_income", [0.01, 0.99], relativeError=0.01
)
inc_lower = income_quantiles[0]
inc_upper = income_quantiles[1]

print(f"  Lower bound (P1)  : ₹{inc_lower:>12,.2f}")
print(f"  Upper bound (P99) : ₹{inc_upper:>12,.2f}")

customers = customers.withColumn(
    "monthly_income",
    F.when(F.col("monthly_income") < inc_lower, inc_lower)
     .when(F.col("monthly_income") > inc_upper, inc_upper)
     .otherwise(F.col("monthly_income"))
)

print("  Income capping applied ✅")

# Single lightweight verification — no shuffle
print("\nIncome stats after capping:")
customers.select(
    F.min("monthly_income").alias("min"),
    F.percentile_approx("monthly_income", 0.5, 100).alias("median"),
    F.max("monthly_income").alias("max")
).show()

STEP 6 — OUTLIER CAPPING

── Transaction Amount Capping ──
  Lower bound (P0.1)  : ₹       12.53
  Upper bound (P99.9) : ₹5,456,127.90


Py4JJavaError: An error occurred while calling o468.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 111.0 failed 1 times, most recent failure: Lost task 0.0 in stage 111.0 (TID 79) (192.168.1.44 executor driver): java.io.FileNotFoundException: C:\Users\dsp96\AppData\Local\Temp\blockmgr-31fb2361-9d17-44d8-abde-866212847d8e\02\temp_shuffle_f928552d-32d9-49ad-8a79-46411bd34b6f (The system cannot find the path specified)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.BypassMergeSortS

In [ ]:
print("STEP 7 — ADD DERIVED COLUMNS AND SAVE\n")

# ── Add account age in months (as of history period end) ──────
customers = customers.withColumn(
    "account_age_months",
    F.round(
        F.months_between(
            F.lit("2023-12-31").cast(DateType()),
            F.col("account_open_date")
        ), 1
    )
)

# ── Add income band ───────────────────────────────────────────
customers = customers.withColumn(
    "income_band",
    F.when(F.col("monthly_income") <  15000,  "very_low")
     .when(F.col("monthly_income") <  40000,  "low")
     .when(F.col("monthly_income") < 100000,  "mid")
     .when(F.col("monthly_income") < 300000,  "high")
     .otherwise("very_high")
)

# ── Verify final null counts ───────────────────────────────────
print("Final null counts — customers:")
customers.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["age","gender","monthly_income","cibil_score",
              "account_open_date","account_age_months"]
]).show()

# ── Save to CSV using pandas (bypasses Hadoop on Windows) ────
import pandas as pd

PROCESSED_PATH = "../data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Saving cleaned data to CSV (using pandas)...")

# Convert to pandas and save as CSV (customers table is small enough)
customers.toPandas().to_csv(
    f"{PROCESSED_PATH}/customers_clean.csv", index=False
)

# For large tables, save with coalesce to single partition then manually write
# This avoids Hadoop issues and memory issues
print("✅ customers_clean.csv saved")

# Products table (smaller, can use pandas)
products.toPandas().to_csv(
    f"{PROCESSED_PATH}/products_clean.csv", index=False
)
print("✅ products_clean.csv saved")

# Note: txn_history and clv_labels are too large for single-machine pandas conversion
# In production, you would use distributed storage (S3, HDFS, etc.) with .write.parquet()
print("\n⚠️  Note: Large tables (txn_history, clv_labels) require distributed storage")
print("   For now, they remain in PySpark DataFrames in memory")

# Verify saved files can be read back
customers_check = pd.read_csv(f"{PROCESSED_PATH}/customers_clean.csv")
print(f"\nVerification — customers_clean rows: {len(customers_check):,}")
print(f"Columns: {len(customers_check.columns)}")

STEP 7 — ADD DERIVED COLUMNS AND SAVE

Final null counts — customers:
+---+------+--------------+-----------+-----------------+------------------+
|age|gender|monthly_income|cibil_score|account_open_date|account_age_months|
+---+------+--------------+-----------+-----------------+------------------+
|  0|     0|             0|          0|                0|                 0|
+---+------+--------------+-----------+-----------------+------------------+

Saving cleaned data to CSV (using pandas)...


c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


✅ customers_clean.csv saved


c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\dsp96\Desktop\realworld-ds-ml\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


✅ products_clean.csv saved

⚠️  Note: Large tables (txn_history, clv_labels) require distributed storage
   For now, they remain in PySpark DataFrames in memory

Verification — customers_clean rows: 50,000
Columns: 21


In [ ]:
import os
import shutil
import glob

PROCESSED_PATH = "../data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

def save_spark_df(df, name, processed_path):
    """
    Save Spark DataFrame to CSV without coalesce.
    Writes multiple part files — avoids shuffle crash.
    Then merges part files in Python (no Spark needed).
    """
    temp_dir   = os.path.abspath(
        os.path.join(processed_path, f"_{name}_tmp")
    )
    final_path = os.path.abspath(
        os.path.join(processed_path, f"{name}.csv")
    )

    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    # Write WITHOUT coalesce — no shuffle, just writes
    # each partition directly to a part file
    (
        df.write
        .mode("overwrite")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .csv(temp_dir)
    )

    # Merge part files in Python
    part_files = sorted(glob.glob(
        os.path.join(temp_dir, "part-*.csv")
    ))

    if not part_files:
        print(f"  ⚠️  No part files found for {name}")
        return

    if os.path.exists(final_path):
        os.remove(final_path)

    with open(final_path, "wb") as out:
        for i, pf in enumerate(part_files):
            with open(pf, "rb") as pf_in:
                if i > 0:
                    # Skip header line for all files after first
                    pf_in.readline()
                shutil.copyfileobj(pf_in, out)

    shutil.rmtree(temp_dir)

    size_mb = os.path.getsize(final_path) / 1024 / 1024
    print(f"  ✅ {name}.csv  →  "
          f"{len(part_files)} parts merged  |  {size_mb:.1f} MB")

print("Saving large tables...\n")
save_spark_df(txn_history, "transactions_clean", PROCESSED_PATH)
save_spark_df(clv_labels,  "clv_labels_clean",   PROCESSED_PATH)

print("\nProcessed folder:")
for f in sorted(os.listdir(PROCESSED_PATH)):
    fp = os.path.join(PROCESSED_PATH, f)
    if os.path.isfile(fp):
        size = os.path.getsize(fp) / 1024 / 1024
        print(f"  {f:<38} {size:>8.1f} MB")

Saving large tables with Spark parallel writes...



NameError: name 'txn_history' is not defined